# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fiona-ali/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring. **Decision:** which eligible pages should an editor inspect first for a possible refresh?

This notebook trains a logistic regression model to rank pages against the starter dataset's observed decline proxy. The proxy is `trend_direction == "down"`, which is computed from the most recent 30-day impressions compared with the prior 30 days. The supplied snapshot does not provide a later independent outcome window. Therefore, the scores below measure agreement with an observed trailing-window proxy, not prediction of future decline and not the causal effect of refreshing a page.

The repository's 30,000-row starter CSV is used. The eligibility rule (at least 100 impressions) follows ML-07. The labels are kept separate from model inputs.


## 1. Method choice and why

I use **Logistic Regression** because the task asks for a ranked review list and the model's predicted probability can serve as a simple ranking score. It is a readable first learned model; this is not a reason to claim the probabilities are calibrated.

The model inputs are limited to content metadata and properties available before any refresh decision: search volume, competition, CPC, content type, intent, word count, character count, content age, and days since last update. Numeric missing values are median-imputed with missingness indicators; categorical missing values get an explicit category. Client and content IDs are never model features.

I exclude `trend_direction`, `trend_pct`, the label, both 30-day impression windows, and all 90-day performance totals/rates. The label is constructed directly from recent and prior 30-day impressions, so those columns (and aggregates that contain them) would expose the answer period. This conservative input set may have less ranking power, but it avoids giving the model the label formula.

**Research reading:** the linked [FlyRank State of AI-Driven SEO report (April 2026)](https://state-of-seo-2026.flyrank.ai/) says its headline evidence prioritizes direct aggregate comparisons and that its ML analyses are exploratory; it also identifies content age as a confounder and says its pattern study is not causal. I apply the same restraint here: the starter slice and trailing proxy support a method exercise, not a deployment claim.


In [1]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
import sklearn

# Find the repository data in a local clone or in Colab. If only the notebook is present,
# clone the public repository so the same checked-in starter CSV is used.
DATA_REL = Path("data/raw/content_refresh_anonymized.csv")
repo_candidates = [Path.cwd(), Path.cwd() / "flyrank-ml-internship", Path("/content/flyrank-ml-internship")]
REPO_ROOT = next((p for p in repo_candidates if (p / DATA_REL).exists()), None)
if REPO_ROOT is None:
    clone_path = Path("/tmp/flyrank-ml-internship")
    if not (clone_path / DATA_REL).exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Fiona-ali/flyrank-ml-internship.git", str(clone_path)], check=True)
    REPO_ROOT = clone_path
DATA_PATH = REPO_ROOT / DATA_REL

df = pd.read_csv(DATA_PATH)
required = {"content_id", "client_id", "trend_direction", "impressions_90d", "search_volume", "competition", "cpc", "content_type", "main_intent", "word_count", "char_count", "content_age_days", "days_since_last_update"}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Starter data is missing required columns: {missing}")
if df.shape != (30_000, 44):
    raise ValueError(f"Unexpected starter data shape: {df.shape}; expected 30,000 rows × 44 columns")

# Outcome proxy is derived only for evaluation. Never put it in X.
df["decline_proxy"] = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)
eligible = df.loc[pd.to_numeric(df["impressions_90d"], errors="coerce") >= 100].copy().reset_index(drop=True)
print(f"Dataset: {len(df):,} rows × 44 columns; {df['client_id'].nunique()} clients")
print(f"Eligible after the ML-07 floor (impressions_90d >= 100): {len(eligible):,} rows")
print(f"Observed decline-proxy base rate in eligible rows: {eligible['decline_proxy'].mean():.3f}")
print(f"Versions: pandas {pd.__version__}; scikit-learn {sklearn.__version__}")
print("This target is a trailing-window proxy, not a future outcome.")


Dataset: 30,000 rows × 44 columns; 32 clients
Eligible after the ML-07 floor (impressions_90d >= 100): 22,006 rows
Observed decline-proxy base rate in eligible rows: 0.598
Versions: pandas 2.2.3; scikit-learn 1.8.0
This target is a trailing-window proxy, not a future outcome.


## 2. Split design

I hold out **25% of eligible rows by client**, with a fixed seed (`42`). `GroupShuffleSplit` keeps each held-out client's pages out of training, which checks whether the model's patterns transfer to clients it did not train on. The split is row-based within whole clients, so the actual test fraction can differ slightly from exactly 25%.

The rule baseline is recomputed on these exact held-out rows. ML-07 reported full-eligible-slice descriptive numbers rather than a client-held-out test, so those headline values are not directly comparable to this test split. The table below is the fair within-ML-08 comparison.


In [2]:
from sklearn.model_selection import GroupShuffleSplit

NUMERIC_FEATURES = ["search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days", "days_since_last_update"]
CATEGORICAL_FEATURES = ["content_type", "main_intent"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FORBIDDEN = {"content_id", "client_id", "trend_direction", "trend_pct", "decline_proxy", "is_declining_label",
             "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
             "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
             "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"}
assert not (set(FEATURES) & FORBIDDEN), "A label-derived, period-overlapping, or ID column entered the feature list."

X = eligible[FEATURES].copy()
y = eligible["decline_proxy"].to_numpy(dtype=int)
groups = eligible["client_id"].to_numpy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
heldout_clients = set(groups[test_idx])
assert heldout_clients.isdisjoint(set(groups[train_idx]))
print(f"Train: {len(train_idx):,} rows; test: {len(test_idx):,} rows")
print(f"Train/test clients: {len(set(groups[train_idx]))}/{len(heldout_clients)} (no client overlap)")
print(f"Test-set decline-proxy base rate: {y_test.mean():.3f}")
print(f"Input columns ({len(FEATURES)}): {', '.join(FEATURES)}")


Train: 17,396 rows; test: 4,610 rows
Train/test clients: 22/8 (no client overlap)
Test-set decline-proxy base rate: 0.577
Input columns (9): search_volume, competition, cpc, word_count, char_count, content_age_days, days_since_last_update, content_type, main_intent


## 3. Train + compare against the Week-4 baseline

The baseline is the frozen ML-07 rule: `log1p(impressions_90d) × min(days_since_last_update, 180) / 180`. I compute that score on the same client-held-out test rows and compare it with the model using base rate, Precision@10, Precision@50, average precision, and ROC AUC. Precision@K matches the practical “which pages first?” decision; average precision and ROC AUC summarize ranking over the full test slice.

**Baseline limitation:** ML-07 uses a 90-day impressions total that overlaps the label's two 30-day windows. I retain that rule only as the requested historical comparator and disclose the overlap. It is not a leakage-safe estimate of future performance. The trained model excludes those overlapping fields.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("numeric", numeric_pipe, NUMERIC_FEATURES),
    ("categorical", categorical_pipe, CATEGORICAL_FEATURES),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

# Compute the frozen Week-4 score on precisely the same held-out rows.
rule_scores = (
    np.log1p(pd.to_numeric(eligible.iloc[test_idx]["impressions_90d"], errors="coerce").clip(lower=0).to_numpy())
    * pd.to_numeric(eligible.iloc[test_idx]["days_since_last_update"], errors="coerce").clip(lower=0, upper=180).to_numpy()
    / 180.0
)

def ranking_metrics(name, scores):
    order = np.argsort(-np.asarray(scores), kind="mergesort")
    return {
        "Ranking": name,
        "Precision@10": float(y_test[order[:min(10, len(order))]].mean()),
        "Precision@50": float(y_test[order[:min(50, len(order))]].mean()),
        "Average precision": float(average_precision_score(y_test, scores)),
        "ROC AUC": float(roc_auc_score(y_test, scores)),
    }

results = pd.DataFrame([
    {"Ranking": "Base rate (random-order reference)", "Precision@10": float(y_test.mean()), "Precision@50": float(y_test.mean()), "Average precision": float(y_test.mean()), "ROC AUC": 0.5},
    ranking_metrics("Week-4 rule (same held-out rows)", rule_scores),
    ranking_metrics("Logistic Regression", model_scores),
])
print("Evaluation: 4,610 held-out rows from 8 clients; same rows for each ranking")
display(results.round(3))


Evaluation: 4,610 held-out rows from 8 clients; same rows for each ranking


,Ranking,Precision@10,Precision@50,Average precision,ROC AUC
0,Base rate (random-order reference),0.577,0.577,0.577,0.500
1,Week-4 rule (same held-out rows),0.500,0.380,0.521,0.418
2,Logistic Regression,0.600,0.720,0.632,0.571


## 4. Errors and interpretation

I inspect model-level errors at a 0.50 threshold and show high-scoring false positives as anonymous feature profiles (no IDs, URLs, client names, or query text). The threshold is used only to describe errors; ranking still uses predicted probabilities. Permutation importance measures how much held-out average precision changes when a feature is shuffled. Correlated features can share or dilute importance, so this is a diagnostic, not causal attribution.


In [4]:
from sklearn.inspection import permutation_importance

predicted = (model_scores >= 0.50).astype(int)
false_positive_positions = np.flatnonzero((predicted == 1) & (y_test == 0))
false_negative_positions = np.flatnonzero((predicted == 0) & (y_test == 1))
print(f"At a 0.50 threshold: {int((predicted == y_test).sum()):,}/{len(y_test):,} correct; {len(false_positive_positions):,} false positives; {len(false_negative_positions):,} false negatives.")

# Three concrete mistakes, described only through the permitted feature fields.
if len(false_positive_positions):
    fp_order = false_positive_positions[np.argsort(-model_scores[false_positive_positions], kind="mergesort")[:3]]
    fp_profiles = X_test.iloc[fp_order].copy().reset_index(drop=True)
    fp_profiles.insert(0, "predicted_probability", model_scores[fp_order].round(3))
    fp_profiles["observed_decline_proxy"] = y_test[fp_order]
    print("Highest-scoring false-positive profiles (feature values only):")
    display(fp_profiles)
else:
    print("No false positives at a 0.50 threshold in this split.")

# Show the three strongest missed decline-proxy cases, also without identifiers.
if len(false_negative_positions):
    fn_order = false_negative_positions[np.argsort(model_scores[false_negative_positions], kind="mergesort")[-3:][::-1]]
    fn_profiles = X_test.iloc[fn_order].copy().reset_index(drop=True)
    fn_profiles.insert(0, "predicted_probability", model_scores[fn_order].round(3))
    fn_profiles["observed_decline_proxy"] = y_test[fn_order]
    print("Three false-negative profiles closest to the threshold:")
    display(fn_profiles)

importance = permutation_importance(model, X_test, y_test, scoring="average_precision", n_repeats=5, random_state=42, n_jobs=-1)
importance_table = pd.DataFrame({
    "feature": FEATURES,
    "mean_AP_drop_when_shuffled": importance.importances_mean,
    "std": importance.importances_std,
}).sort_values("mean_AP_drop_when_shuffled", ascending=False).reset_index(drop=True)
print("Permutation importance on held-out average precision (positive means shuffling reduced AP):")
display(importance_table.round(4))


At a 0.50 threshold: 2,632/4,610 correct; 1,188 false positives; 790 false negatives.
Highest-scoring false-positive profiles (feature values only):
Three false-negative profiles closest to the threshold:
Permutation importance on held-out average precision (positive means shuffling reduced AP):


,predicted_probability,search_volume,competition,cpc,word_count,char_count,content_age_days,days_since_last_update,content_type,main_intent,observed_decline_proxy
0,0.863,0.0,0.0,0.0,2528.0,18569.0,126,20,keyword article,NaN,0
1,0.785,NaN,NaN,NaN,1094.0,7718.0,305,304,keyword article,NaN,0
2,0.781,NaN,NaN,NaN,1397.0,9784.0,301,301,keyword article,NaN,0


,predicted_probability,search_volume,competition,cpc,word_count,char_count,content_age_days,days_since_last_update,content_type,main_intent,observed_decline_proxy
0,0.5,20.0,0.00,0.00,3152.0,19766.0,421,7,keyword article,commercial,1
1,0.5,140.0,0.96,0.47,2891.0,18179.0,421,7,keyword article,transactional,1
2,0.5,1300.0,0.00,0.00,3207.0,20731.0,445,7,keyword article,informational,1


,feature,mean_AP_drop_when_shuffled,std
0,content_age_days,0.0558,0.0061
1,main_intent,0.0154,0.0046
2,word_count,0.0045,0.0029
3,days_since_last_update,-0.0001,0.0037
4,content_type,-0.0005,0.0023
5,competition,-0.0014,0.0019
6,search_volume,-0.0015,0.0019
7,cpc,-0.0016,0.0022
8,char_count,-0.0035,0.0015


### Interpretation and limits

On this fixed split, the logistic model reached **Precision@50 = 0.720**, versus **0.380** for the Week-4 rule, with a held-out base rate of **0.577**. At K=10 the results were **0.600** and **0.500**, respectively. Average precision was **0.632** for the model and **0.521** for the rule; ROC AUC was **0.571** and **0.418**. The result is a directional improvement over the rule on this split, not proof of future prediction. In particular, the baseline's 90-day impressions overlap the trailing label windows.

Permutation importance was largest for **content age** (mean held-out AP drop 0.0558), followed by **main intent** (0.0154) and **word count** (0.0045). This describes how the fitted model uses these features on this sample; it does not show that changing age, intent, or article length would cause a better outcome. Individual errors show the same limit: some high-scoring pages did not have the observed decline proxy, while some near-threshold pages did.

The decision supported by this notebook is **editor review prioritization**, not automatic refreshing. A metadata-only model is intentionally tested because the available target proxy is defined from the same trailing performance windows that appear in traffic features. The rule baseline is shown for continuity with ML-07, but its 90-day impression input overlaps those label windows and therefore is not evidence of future predictive skill.

The held-out-client result is a single fixed split on a 30,000-row teaching extract, not the full warehouse. Performance can shift with another set of clients, another time period, or a target defined from a future outcome window. The linked FlyRank report likewise warns that its machine-learning analyses are exploratory and that content age is a confounder. Before using this operationally, define a future label window and validate on a later time period with a client-aware design.


## Self-check

- [x] Method choice and features are explained; output is a ranked review score.
- [x] Split is grouped by client; held-out clients do not appear in training.
- [x] Week-4 rule and model are evaluated on the exact same held-out rows and metrics.
- [x] The target source and the baseline's overlap with its trailing label windows are disclosed.
- [x] Label-derived fields, traffic-window features, IDs, and query/content identities are excluded from model inputs.
- [x] Error profiles and permutation importance are computed from the run, not invented.
- [x] Notebook executed top to bottom with fixed random seeds. The starter CSV stays out of any new commit.
- [x] Commit this notebook under `work/notebooks/`, then submit the repo URL on the ML-08 card.
